# Lab 3: Agentic Scientific Review Research

Single canonical Stage 0–8 flow: baseline vs custom MAS orchestration, Ollama synthesis/evaluation (local), MLflow-backed cache replay (Colab), controlled ablations, and rich in-notebook final report.


## Stage 0 - Setup and Reproducibility

**Control question:** Is this setup sufficient for deterministic replay of all stages?


In [1]:
import sys
import os
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPLAY_ONLY = IN_COLAB

if IN_COLAB:
    repo_root = Path("/content/itmo-ml-2025")
    req_file = repo_root / "dl" / "lab-3" / "requirements.txt"
    if req_file.exists():
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_file)])
    cache_zip = repo_root / "dl" / "lab-3" / "results_cache.zip"
    cache_dir = repo_root / "dl" / "lab-3" / "results_cache"
    if cache_zip.exists():
        import zipfile
        cache_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(cache_zip, "r") as zf:
            zf.extractall(cache_dir)
        print(f"Restored cache from {cache_zip}")
    print("Colab REPLAY_ONLY=True — load artifacts from cache/MLflow, skip Ollama on hits")
else:
    print("Local mode — Ollama on cache miss, all runs logged to MLflow")


Local mode — Ollama on cache miss, all runs logged to MLflow


In [2]:
import json
import time
import math
import pickle
import random
import hashlib
import platform
import re
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
import torch
import mlflow
import plotly.express as px
from mlflow.tracking import MlflowClient

ROOT_DIR = Path.cwd()
RESULTS_CACHE_DIR = ROOT_DIR / "results_cache"
RESULTS_CACHE_DIR.mkdir(parents=True, exist_ok=True)

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434")
OLLAMA_MODEL = 'llama3.1'
OLLAMA_TEMPERATURE = 0.0
MLFLOW_EXPERIMENT = "lab3_mas_research"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

TRACKING_DIR = RESULTS_CACHE_DIR / "mlruns"
mlflow.set_tracking_uri(TRACKING_DIR.as_uri())
mlflow.set_experiment(MLFLOW_EXPERIMENT)
MLFLOW_CLIENT = MlflowClient()

RUNTIME_DISCLOSURE = {
    "provider": "ollama",
    "model": OLLAMA_MODEL,
    "base_url": OLLAMA_BASE_URL,
    "temperature": OLLAMA_TEMPERATURE,
    "platform": platform.platform(),
    "python": platform.python_version(),
    "replay_only": REPLAY_ONLY,
}
RUNTIME_DISCLOSURE


/Users/lopatenko/Desktop/itmo/itmo-ml-2025/.venv/lib/python3.14/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


{'provider': 'ollama',
 'model': 'llama3.1',
 'base_url': 'http://127.0.0.1:11434',
 'temperature': 0.0,
 'platform': 'macOS-14.6.1-arm64-arm-64bit-Mach-O',
 'python': '3.14.3',
 'replay_only': False}

### Model and Runtime Disclosure

- **Provider:** local Ollama (`OLLAMA_BASE_URL`, default `http://127.0.0.1:11434`).
- **Model:** `OLLAMA_MODEL` (default `llama3.2:3b`); quantization/hardware depend on your Ollama pull.
- **Colab:** `REPLAY_ONLY=True` after restoring `results_cache.zip` — no Ollama on cache hit.
- **Reproducibility:** same topics, `ExperimentConfig` cache keys, MLflow artifacts.


## Stage 1 - Schema, Tools, State, Cache, and Ollama Client

**Control question:** Does state trace explain transitions and stop/fallback reasons?


In [3]:
OUTPUT_FIELDS = ["definition", "main_approaches", "key_works", "applications", "limitations", "used_sources"]

@dataclass(frozen=True)
class LLMConfig:
    model_name: str = OLLAMA_MODEL
    temperature: float = OLLAMA_TEMPERATURE
    max_tokens: int = 1200

@dataclass(frozen=True)
class ExperimentConfig:
    stage: str
    topic: str
    mode: str
    source_limit: int = 5
    max_steps: int = 6
    use_evaluator: bool = False
    llm: LLMConfig = field(default_factory=LLMConfig)

def cfg_to_cache_key(cfg: ExperimentConfig) -> str:
    return hashlib.sha1(json.dumps(asdict(cfg), sort_keys=True).encode("utf-8")).hexdigest()

def artifact_name(cfg: ExperimentConfig, suffix: str = "") -> str:
    stem = cfg_to_cache_key(cfg)
    return f"{stem}_{suffix}.pkl" if suffix else f"{stem}.pkl"

def cache_path(cfg: ExperimentConfig, suffix: str = "") -> Path:
    return RESULTS_CACHE_DIR / artifact_name(cfg, suffix)

def _safe_metric(v: Any) -> Optional[float]:
    try:
        f = float(v)
        return f if math.isfinite(f) else None
    except Exception:
        return None

def _mlflow_tags(cfg: ExperimentConfig, suffix: str = "") -> Dict[str, str]:
    return {
        "cache_key": cfg_to_cache_key(cfg),
        "suffix": suffix,
        "stage": cfg.stage,
        "mode": cfg.mode,
        "topic": cfg.topic,
        "model": cfg.llm.model_name,
    }

def save_result(result: Dict[str, Any], cfg: ExperimentConfig, suffix: str = "") -> Dict[str, Any]:
    p = cache_path(cfg, suffix)
    with p.open("wb") as f:
        pickle.dump(result, f)
    with mlflow.start_run(run_name=f"{cfg.stage}_{cfg.mode}_{cfg.topic[:20]}"):
        mlflow.set_tags(_mlflow_tags(cfg, suffix))
        mlflow.log_params({
            "stage": cfg.stage, "mode": cfg.mode, "topic": cfg.topic,
            "source_limit": cfg.source_limit, "max_steps": cfg.max_steps,
            "use_evaluator": int(cfg.use_evaluator), "provider": "ollama",
            "ollama_model": OLLAMA_MODEL, "ollama_base_url": OLLAMA_BASE_URL,
        })
        for k, v in result.get("metrics", {}).items():
            fv = _safe_metric(v)
            if fv is not None:
                mlflow.log_metric(k, fv)
        mlflow.log_artifact(str(p), artifact_path="cached_results")
    return result

def _experiment_ids() -> List[str]:
    ids = []
    for name in (MLFLOW_EXPERIMENT, "lab3_mas_research_automas"):
        exp = mlflow.get_experiment_by_name(name)
        if exp is not None:
            ids.append(exp.experiment_id)
    return ids

def _load_result_from_mlflow(cfg: ExperimentConfig, suffix: str = "") -> Optional[Dict[str, Any]]:
    tags = _mlflow_tags(cfg, suffix)
    filt = " and ".join([f"tags.{k} = '{v}'" for k, v in tags.items()])
    for exp_id in _experiment_ids():
        runs = MLFLOW_CLIENT.search_runs([exp_id], filter_string=filt, max_results=1,
                                           order_by=["attributes.start_time DESC"])
        if not runs:
            continue
        local = MLFLOW_CLIENT.download_artifacts(
            runs[0].info.run_id, f"cached_results/{artifact_name(cfg, suffix)}")
        with open(local, "rb") as f:
            obj = pickle.load(f)
        with cache_path(cfg, suffix).open("wb") as f:
            pickle.dump(obj, f)
        return obj
    return None

def load_result(cfg: ExperimentConfig, suffix: str = "") -> Optional[Dict[str, Any]]:
    p = cache_path(cfg, suffix)
    if p.exists():
        with p.open("rb") as f:
            return pickle.load(f)
    return _load_result_from_mlflow(cfg, suffix)

def run_or_load(cfg: ExperimentConfig, supplier, suffix: str = "") -> Dict[str, Any]:
    cached = load_result(cfg, suffix)
    if cached is not None:
        return cached
    if REPLAY_ONLY:
        raise RuntimeError(f"Cache miss in REPLAY_ONLY for topic={cfg.topic!r} mode={cfg.mode}")
    return save_result(supplier(), cfg, suffix)

def schema_coverage(answer: Dict[str, Any]) -> float:
    return sum(1 for f in OUTPUT_FIELDS if bool(answer.get(f))) / len(OUTPUT_FIELDS)

def extract_json_block(text: str) -> Optional[Dict[str, Any]]:
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    m = re.search(r"\{[\s\S]*\}", text)
    if m:
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError:
            return None
    return None

_ollama_available: Optional[bool] = None

def ollama_available() -> bool:
    global _ollama_available
    if REPLAY_ONLY:
        return False
    if _ollama_available is not None:
        return _ollama_available
    try:
        import ollama
        ollama.Client(host=OLLAMA_BASE_URL).list()
        _ollama_available = True
    except Exception:
        _ollama_available = False
    return _ollama_available

def ollama_chat(prompt: str, system: str = "", model: str = OLLAMA_MODEL, temperature: float = OLLAMA_TEMPERATURE) -> Tuple[str, Dict[str, Any]]:
    meta: Dict[str, Any] = {"provider": "ollama", "model": model}
    if not ollama_available():
        return "", meta
    import ollama
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = ollama.Client(host=OLLAMA_BASE_URL).chat(model=model, messages=messages, options={"temperature": temperature})
    text = resp.get("message", {}).get("content", "")
    meta["eval_count"] = resp.get("eval_count")
    meta["prompt_eval_count"] = resp.get("prompt_eval_count")
    return text, meta

def synthesize_with_ollama(topic: str, wiki_text: str, sources: List[Dict[str, Any]], notes: List[str]) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    src_lines = "\n".join(f"- {s.get('title','')} ({s.get('year','')})" for s in sources[:12])
    note_blob = "\n".join(notes[:8])[:4000]
    system = "You are a scientific review assistant. Return ONLY valid JSON."
    prompt = (
        f"Topic: {topic}\nWikipedia excerpt: {wiki_text[:1500]}\nSources:\n{src_lines}\n"
        f"Notes:\n{note_blob}\n\n"
        "Return JSON with keys: definition, main_approaches, key_works, applications, limitations, used_sources."
    )
    text, meta = ollama_chat(prompt, system=system)
    parsed = extract_json_block(text)
    if parsed is None:
        repair_prompt = f"Fix to valid JSON only. Previous output:\n{text[:2000]}"
        text2, meta2 = ollama_chat(repair_prompt, system=system)
        meta["repair"] = True
        meta.update({k: meta2.get(k) for k in ("eval_count", "prompt_eval_count")})
        parsed = extract_json_block(text2)
    if parsed is None:
        return deterministic_answer(topic, wiki_text, sources), {"fallback": "deterministic_synthesis", **meta}
    for f in OUTPUT_FIELDS:
        parsed.setdefault(f, [] if f != "definition" else "")
    return parsed, meta

def deterministic_answer(topic: str, wiki_text: str, sources: List[Dict[str, Any]]) -> Dict[str, Any]:
    lead = wiki_text.split(".")[0].strip() if wiki_text else f"{topic} is a research area."
    return {
        "definition": lead,
        "main_approaches": ["retrieval-guided synthesis", "role-based reasoning", "structured reporting"],
        "key_works": [{"title": s.get("title", ""), "year": s.get("year"), "url": s.get("url", "")} for s in sources[:5]],
        "applications": ["literature review drafting", "research mapping", "evidence discovery"],
        "limitations": ["depends on retrieval quality", "can miss niche sources"],
        "used_sources": [{"title": s.get("title", ""), "url": s.get("url", "")} for s in sources],
    }


In [4]:
def reconstruct_openalex_abstract(inv_idx: Optional[Dict[str, List[int]]]) -> str:
    if not inv_idx:
        return ""
    n = max(max(ps) for ps in inv_idx.values()) + 1
    arr = [""] * n
    for w, ps in inv_idx.items():
        for p in ps:
            arr[p] = w
    return " ".join(t for t in arr if t)

def merge_counters(*cs: Dict[str, int]) -> Dict[str, int]:
    out = {"tool_calls": 0, "tool_errors": 0, "fallback_activations": 0, "timeouts": 0}
    for c in cs:
        for k in out:
            out[k] += int(c.get(k, 0))
    return out

WIKI_USER_AGENT = os.getenv("WIKI_USER_AGENT", "itmo-ml-lab3/1.0 (education; contact: student@itmo.ru)")
WIKI_HEADERS = {"User-Agent": WIKI_USER_AGENT}
TOPIC_WIKI_SLUGS = {
    "neuro-symbolic artificial intelligence": "Neuro-symbolic_AI",
}

def _wiki_summary_by_slug(slug: str, timeout: int = 20):
    enc = requests.utils.quote(slug.replace(" ", "_"), safe="/()")
    return requests.get(
        f"https://en.wikipedia.org/api/rest_v1/page/summary/{enc}",
        timeout=timeout,
        headers=WIKI_HEADERS,
    )

def tool_wikipedia_summary(topic: str, timeout: int = 20):
    c = {"tool_calls": 1, "tool_errors": 0, "fallback_activations": 0, "timeouts": 0}
    try:
        overrides = globals().get("TOPIC_WIKI_SLUGS", {})
        slug_candidates = []
        if topic in overrides:
            slug_candidates.append(overrides[topic])
        slug_candidates.append(topic.replace(" ", "_"))
        for slug in slug_candidates:
            r = _wiki_summary_by_slug(slug, timeout=timeout)
            if r.status_code == 200:
                j = r.json()
                return {
                    "ok": True,
                    "text": j.get("extract", ""),
                    "url": j.get("content_urls", {}).get("desktop", {}).get("page", ""),
                    "wiki_slug": slug,
                }, c
        r = requests.get(
            "https://en.wikipedia.org/w/api.php",
            params={"action": "query", "list": "search", "srsearch": topic, "format": "json", "srlimit": 1},
            timeout=timeout,
            headers=WIKI_HEADERS,
        )
        if r.status_code == 200:
            hits = r.json().get("query", {}).get("search", [])
            if hits:
                title = hits[0]["title"]
                r2 = _wiki_summary_by_slug(title, timeout=timeout)
                if r2.status_code == 200:
                    j = r2.json()
                    return {
                        "ok": True,
                        "text": j.get("extract", ""),
                        "url": j.get("content_urls", {}).get("desktop", {}).get("page", ""),
                        "wiki_slug": title.replace(" ", "_"),
                    }, c
        c["tool_errors"] += 1
        c["fallback_activations"] += 1
        return {"ok": False, "text": "", "url": ""}, c
    except requests.exceptions.Timeout:
        c["tool_errors"] += 1
        c["timeouts"] += 1
        c["fallback_activations"] += 1
        return {"ok": False, "text": "", "url": ""}, c
    except Exception:
        c["tool_errors"] += 1
        c["fallback_activations"] += 1
        return {"ok": False, "text": "", "url": ""}, c

def tool_openalex_search(topic: str, limit: int = 8, timeout: int = 30):
    c = {"tool_calls": 1, "tool_errors": 0, "fallback_activations": 0, "timeouts": 0}
    try:
        r = requests.get(
            "https://api.openalex.org/works",
            params={"search": topic, "per_page": max(1, min(limit, 25)), "sort": "cited_by_count:desc"},
            timeout=timeout,
        )
        if r.status_code != 200:
            c["tool_errors"] += 1
            c["fallback_activations"] += 1
            return [], c
        items = []
        for row in r.json().get("results", []):
            items.append({
                "title": row.get("display_name", ""),
                "year": row.get("publication_year"),
                "citations": row.get("cited_by_count", 0),
                "abstract": reconstruct_openalex_abstract(row.get("abstract_inverted_index")),
                "url": row.get("primary_location", {}).get("landing_page_url") or row.get("doi") or row.get("id"),
            })
        return items, c
    except requests.exceptions.Timeout:
        c["tool_errors"] += 1
        c["timeouts"] += 1
        c["fallback_activations"] += 1
        return [], c
    except Exception:
        c["tool_errors"] += 1
        c["fallback_activations"] += 1
        return [], c

@dataclass
class TraceStep:
    step_id: int
    actor: str
    action: str
    reason: str
    payload: Dict[str, Any]
    state_delta: Dict[str, Any]

@dataclass
class AgentState:
    topic: str
    objective: str
    step: int = 0
    sources: List[Dict[str, Any]] = field(default_factory=list)
    notes: List[str] = field(default_factory=list)
    history: List[TraceStep] = field(default_factory=list)
    final_answer: Dict[str, Any] = field(default_factory=dict)
    status: str = "running"
    stop_reason: str = ""

    def add_step(self, actor: str, action: str, reason: str, payload: Dict[str, Any], prev_sources: int, prev_notes: int):
        self.step += 1
        self.history.append(TraceStep(
            self.step, actor, action, reason, payload,
            {"sources_before": prev_sources, "sources_after": len(self.sources),
             "notes_before": prev_notes, "notes_after": len(self.notes)},
        ))


## Stage 2 - Baseline Pipeline

**Control question:** Is baseline intentionally simpler and faster than MAS?


In [5]:
def run_baseline(cfg: ExperimentConfig) -> Dict[str, Any]:
    state = AgentState(topic=cfg.topic, objective="Produce structured scientific review")
    t0 = time.perf_counter()
    wiki, c1 = tool_wikipedia_summary(cfg.topic)
    ps, pn = len(state.sources), len(state.notes)
    if wiki.get("ok"):
        state.notes.append(wiki.get("text", "")[:1200])
    state.add_step("baseline", "fetch_wikipedia", "obtain general context", {"ok": wiki.get("ok", False)}, ps, pn)
    items, c2 = tool_openalex_search(cfg.topic, limit=cfg.source_limit)
    ps, pn = len(state.sources), len(state.notes)
    state.sources = sorted(items, key=lambda x: (x.get("citations", 0), x.get("year") or 0), reverse=True)[:cfg.source_limit]
    state.add_step("baseline", "search_openalex", "collect top publications", {"found": len(state.sources)}, ps, pn)
    ps, pn = len(state.sources), len(state.notes)
    answer, llm_meta = synthesize_with_ollama(cfg.topic, wiki.get("text", ""), state.sources, state.notes)
    state.final_answer = answer
    state.status = "done"
    state.stop_reason = "one_pass_synthesis"
    state.add_step("baseline", "finalize", "single pass synthesis", {"coverage": schema_coverage(state.final_answer), "llm": llm_meta}, ps, pn)
    rel = merge_counters(c1, c2)
    if llm_meta.get("fallback"):
        rel["fallback_activations"] += 1
    return {
        "topic": cfg.topic,
        "mode": cfg.mode,
        "answer": state.final_answer,
        "trace": [asdict(s) for s in state.history],
        "sources": state.sources,
        "metrics": {
            "latency_sec": time.perf_counter() - t0,
            "steps": len(state.history),
            "schema_coverage": schema_coverage(state.final_answer),
            "source_count": len(state.sources),
            "unnecessary_actions": 0,
            **rel,
        },
    }


## Stage 3 - MAS Pipeline (Supervisor, Planner, Workers)

**Control question:** Does MAS improve completeness/grounding at the cost of steps and latency?


In [6]:
def planner_actions(max_steps: int):
    return [
        ("fetch_wikipedia", "collect broad context"),
        ("search_openalex", "retrieve publications"),
        ("select_sources", "prioritize relevant evidence"),
        ("enrich_notes", "extract concise notes"),
        ("synthesize_answer", "compose final structured answer"),
    ][:max_steps]

def run_mas(cfg: ExperimentConfig) -> Dict[str, Any]:
    state = AgentState(topic=cfg.topic, objective="Produce structured scientific review")
    rel = {"tool_calls": 0, "tool_errors": 0, "fallback_activations": 0, "timeouts": 0}
    t0 = time.perf_counter()
    wiki_cache = {"ok": False, "text": ""}
    for action, reason in planner_actions(cfg.max_steps):
        if action == "fetch_wikipedia":
            ps, pn = len(state.sources), len(state.notes)
            wiki, c = tool_wikipedia_summary(cfg.topic)
            rel = merge_counters(rel, c)
            wiki_cache = wiki
            if wiki.get("ok"):
                state.notes.append(wiki.get("text", "")[:1200])
            state.add_step("worker_context", action, reason, {"ok": wiki.get("ok", False)}, ps, pn)
        elif action == "search_openalex":
            ps, pn = len(state.sources), len(state.notes)
            items, c = tool_openalex_search(cfg.topic, limit=max(cfg.source_limit, 8))
            rel = merge_counters(rel, c)
            state.sources.extend(items)
            state.add_step("worker_retrieval", action, reason, {"retrieved": len(items)}, ps, pn)
        elif action == "select_sources":
            ps, pn = len(state.sources), len(state.notes)
            state.sources = sorted(state.sources, key=lambda x: (x.get("citations", 0), x.get("year") or 0), reverse=True)[:cfg.source_limit]
            state.add_step("worker_filter", action, reason, {"selected": len(state.sources)}, ps, pn)
        elif action == "enrich_notes":
            ps, pn = len(state.sources), len(state.notes)
            for s in state.sources:
                txt = (s.get("abstract") or "").strip()
                if txt:
                    state.notes.append(f"{s.get('title', '')}: {txt[:300]}")
            state.add_step("worker_notes", action, reason, {"notes": len(state.notes)}, ps, pn)
        elif action == "synthesize_answer":
            ps, pn = len(state.sources), len(state.notes)
            answer, llm_meta = synthesize_with_ollama(cfg.topic, wiki_cache.get("text", ""), state.sources, state.notes)
            state.final_answer = answer
            if llm_meta.get("fallback"):
                rel["fallback_activations"] += 1
            state.add_step("worker_writer", action, reason, {"coverage": schema_coverage(state.final_answer), "llm": llm_meta}, ps, pn)
        ps, pn = len(state.sources), len(state.notes)
        stop = bool(state.final_answer) and schema_coverage(state.final_answer) >= 0.95
        why = "schema_complete" if stop else "continue"
        if not stop and action == "search_openalex" and len(state.sources) == 0:
            rel["fallback_activations"] += 1
            why = "fallback_no_reliable_sources"
            state.status = "done"
            state.stop_reason = why
            state.add_step("supervisor", "check_stop", "fallback due to empty retrieval", {"stop": True, "why": why}, ps, pn)
            break
        state.add_step("supervisor", "check_stop", "control loop", {"stop": stop, "why": why}, ps, pn)
        if stop:
            state.status = "done"
            state.stop_reason = why
            break
    if not state.final_answer:
        state.final_answer, _ = synthesize_with_ollama(cfg.topic, wiki_cache.get("text", ""), state.sources, state.notes)
        if not state.stop_reason:
            state.stop_reason = "step_budget_reached"
    unnecessary = sum(1 for s in state.history if s.action == "check_stop" and not s.payload.get("stop", False))
    return {
        "topic": cfg.topic,
        "mode": cfg.mode,
        "answer": state.final_answer,
        "trace": [asdict(s) for s in state.history],
        "sources": state.sources,
        "metrics": {
            "latency_sec": time.perf_counter() - t0,
            "steps": len(state.history),
            "schema_coverage": schema_coverage(state.final_answer),
            "source_count": len(state.sources),
            "unnecessary_actions": unnecessary,
            **rel,
        },
    }


## Stage 4 - Evaluator Module

**Control question:** Are evaluator scores stable and tied to rubric dimensions?


In [7]:
def evaluate_answer_deterministic(topic: str, answer: Dict[str, Any], sources: List[Dict[str, Any]]) -> Dict[str, Any]:
    cov = schema_coverage(answer)
    src_n = len(sources)
    rubric = {
        "correctness": round(min(5.0, 2.0 + 3.0 * cov), 3),
        "groundedness": round(min(5.0, 1.0 + 0.6 * src_n), 3),
        "completeness": round(min(5.0, 1.0 + 4.0 * cov), 3),
        "source_consistency": round(min(5.0, 1.0 + 0.5 * len(answer.get("used_sources", []))), 3),
        "field_coverage": round(min(5.0, 5.0 * cov), 3),
    }
    rubric["overall"] = round(sum(rubric.values()) / len(rubric), 3)
    return {"topic": topic, "rubric": rubric, "method": "deterministic"}

def evaluate_with_ollama(topic: str, answer: Dict[str, Any], sources: List[Dict[str, Any]], notes: List[str]) -> Dict[str, Any]:
    if not ollama_available():
        out = evaluate_answer_deterministic(topic, answer, sources)
        out["method"] = "deterministic_fallback"
        return out
    prompt = (
        f"Evaluate the scientific review for topic: {topic}\n"
        f"Answer JSON: {json.dumps(answer, ensure_ascii=False)[:3500]}\n"
        f"Sources count: {len(sources)}\n"
        "Return ONLY JSON rubric with floats 1-5: correctness, groundedness, completeness, "
        "coverage_of_required_fields, source_consistency, overall."
    )
    text, meta = ollama_chat(prompt, system="You are a strict scientific reviewer.")
    parsed = extract_json_block(text)
    if parsed is None:
        out = evaluate_answer_deterministic(topic, answer, sources)
        out["method"] = "deterministic_fallback"
        out["ollama_meta"] = meta
        return out
    rubric = {
        "correctness": float(parsed.get("correctness", 3)),
        "groundedness": float(parsed.get("groundedness", 3)),
        "completeness": float(parsed.get("completeness", 3)),
        "field_coverage": float(parsed.get("coverage_of_required_fields", parsed.get("field_coverage", 3))),
        "source_consistency": float(parsed.get("source_consistency", 3)),
        "overall": float(parsed.get("overall", 3)),
    }
    rubric["overall"] = round(rubric["overall"], 3)
    return {"topic": topic, "rubric": rubric, "method": "ollama", "ollama_meta": meta}

def attach_evaluation(result: Dict[str, Any], cfg: ExperimentConfig) -> Dict[str, Any]:
    if cfg.use_evaluator:
        ev = evaluate_with_ollama(result["topic"], result["answer"], result.get("sources", []), [])
    else:
        ev = evaluate_answer_deterministic(result["topic"], result["answer"], result.get("sources", []))
    out = dict(result)
    out["evaluation"] = ev
    m = dict(out.get("metrics", {}))
    for k, v in ev["rubric"].items():
        m[f"eval_{k}"] = v
    m["evaluator_overall"] = ev["rubric"]["overall"]
    m["eval_field_coverage"] = ev["rubric"].get("field_coverage", ev["rubric"].get("coverage_of_required_fields"))
    out["metrics"] = m
    return out


## Stage 5 - Controlled Experiments

**Control question:** Is one factor changed per ablation branch?

### Topic design (high-contrast set)

| Topic | Intended stress on metrics |
|-------|----------------------------|
| Constitutional AI | Alignment nuance; evaluator catches weak grounding |
| causal inference | Cross-domain OpenAlex noise; MAS source filtering |
| meta-learning | Ambiguous top citations; baseline over-generalizes |
| neuro-symbolic artificial intelligence | Multi-paradigm synthesis; methods/limitations |
| federated learning | Applied constraints; process vs quality tradeoff |
| mixture of experts | Architecture-heavy approaches; schema coverage |
| knowledge graph embedding | Structured key_works; source consistency |
| reinforcement learning from human feedback | Anchor (rich literature); fair control |

### Cache invalidation

`ExperimentConfig` cache keys include `topic`. After changing this list, old pickles in `results_cache/` no longer match — expect `Cache hits: 0/72` until you re-run. Archive old `.pkl` files if needed; publish a new `results_cache.zip` for Colab.


In [8]:
TOPICS = [
    "Constitutional AI",
    "causal inference",
    "meta-learning",
    "neuro-symbolic artificial intelligence",
    "federated learning",
    "mixture of experts",
    "knowledge graph embedding",
    "reinforcement learning from human feedback",
]

TOPIC_DESIGN_ROLES = {
    "Constitutional AI": "alignment / evaluator grounding",
    "causal inference": "cross-domain retrieval noise",
    "meta-learning": "ambiguous highly-cited papers",
    "neuro-symbolic artificial intelligence": "multi-paradigm synthesis",
    "federated learning": "applied limitations",
    "mixture of experts": "architecture-heavy methods",
    "knowledge graph embedding": "structured key_works stress",
    "reinforcement learning from human feedback": "anchor control",
}

def validate_topics(topics: List[str]) -> pd.DataFrame:
    rows = []
    for topic in topics:
        wiki, _ = tool_wikipedia_summary(topic)
        items, _ = tool_openalex_search(topic, limit=5)
        rows.append({
            "topic": topic,
            "wiki_ok": bool(wiki.get("ok")),
            "openalex_hits": len(items),
            "design_role": TOPIC_DESIGN_ROLES.get(topic, ""),
        })
    return pd.DataFrame(rows)

topic_preflight_df = validate_topics(TOPICS)
topic_preflight_df


,topic,wiki_ok,openalex_hits,design_role
0,Constitutional AI,True,5,alignment / evaluator grounding
1,causal inference,True,5,cross-domain retrieval noise
2,meta-learning,True,5,ambiguous highly-cited papers
3,neuro-symbolic artificial intelligence,True,5,multi-paradigm synthesis
4,federated learning,True,5,applied limitations
5,mixture of experts,True,5,architecture-heavy methods
6,knowledge graph embedding,True,5,structured key_works stress
7,reinforcement learning from human feedback,True,5,anchor control


In [9]:
BASE_LLM_CFG = LLMConfig()

def run_branch(cfg: ExperimentConfig) -> Dict[str, Any]:
    out = run_baseline(cfg) if cfg.mode == "baseline" else run_mas(cfg)
    return attach_evaluation(out, cfg)

def build_grid() -> List[ExperimentConfig]:
    topics = TOPICS
    grid = []
    for topic in topics:
        grid.append(ExperimentConfig(stage="exp_main", topic=topic, mode="baseline", source_limit=5, max_steps=4, use_evaluator=False, llm=BASE_LLM_CFG))
        grid.append(ExperimentConfig(stage="exp_main", topic=topic, mode="mas", source_limit=5, max_steps=6, use_evaluator=False, llm=BASE_LLM_CFG))
        grid.append(ExperimentConfig(stage="exp_main", topic=topic, mode="mas", source_limit=5, max_steps=6, use_evaluator=True, llm=BASE_LLM_CFG))
        for sl in (3, 5, 8):
            grid.append(ExperimentConfig(stage="ablation_sources", topic=topic, mode="mas", source_limit=sl, max_steps=6, use_evaluator=True, llm=BASE_LLM_CFG))
        for ms in (4, 6, 8):
            grid.append(ExperimentConfig(stage="ablation_steps", topic=topic, mode="mas", source_limit=5, max_steps=ms, use_evaluator=True, llm=BASE_LLM_CFG))
    return grid

def flatten_result(cfg: ExperimentConfig, result: Dict[str, Any]) -> Dict[str, Any]:
    row = {
        "topic": cfg.topic, "stage": cfg.stage, "mode": cfg.mode,
        "source_limit": cfg.source_limit, "max_steps": cfg.max_steps,
        "use_evaluator": cfg.use_evaluator, "cache_key": cfg_to_cache_key(cfg),
    }
    row.update(result.get("metrics", {}))
    return row

def variant_label(row: Dict[str, Any]) -> str:
    if row["mode"] == "baseline":
        return "baseline"
    return "mas_with_evaluator" if row.get("use_evaluator") else "mas"

grid = build_grid()
hits = sum(load_result(cfg) is not None for cfg in grid)
print(f"Cache hits: {hits}/{len(grid)} | REPLAY_ONLY={REPLAY_ONLY} | Ollama available={ollama_available()}")
rows = []
for cfg in grid:
    rows.append(flatten_result(cfg, run_or_load(cfg, lambda c=cfg: run_branch(c))))
results_df = pd.DataFrame(rows)
results_df["variant"] = results_df.apply(variant_label, axis=1)
results_df.head()


Cache hits: 72/72 | REPLAY_ONLY=False | Ollama available=False


,topic,stage,mode,source_limit,max_steps,use_evaluator,cache_key,latency_sec,steps,schema_coverage,...,fallback_activations,timeouts,eval_correctness,eval_groundedness,eval_completeness,eval_source_consistency,eval_field_coverage,eval_overall,evaluator_overall,variant
0,Constitutional AI,exp_main,baseline,5,4,False,f07376326e307c3f27b2edc6a9ae08aca3271c0b,15.841711,3,0.666667,...,0,0,4.0,4.0,3.667,3.5,3.333,3.7,3.7,baseline
1,Constitutional AI,exp_main,mas,5,6,False,497d4776edcefc4e447dd6390bfc54f3fd69a20f,15.636610,10,0.833333,...,0,0,4.5,4.0,4.333,3.5,4.167,4.1,4.1,mas
2,Constitutional AI,exp_main,mas,5,6,True,854dbc524e29cf4c5823e35aa326c35cf257af6a,14.697098,10,0.833333,...,0,0,2.0,3.0,1.500,2.0,4.000,2.5,2.5,mas_with_evaluator
3,Constitutional AI,ablation_sources,mas,3,6,True,aed291fe31c0aa2cef609c663887ef4f0e94f187,11.145582,10,0.666667,...,0,0,2.0,3.0,1.000,5.0,4.000,2.8,2.8,mas_with_evaluator
4,Constitutional AI,ablation_sources,mas,5,6,True,f89d2ff26532ec1c407438785e38620a9af513ce,16.121429,10,0.833333,...,0,0,2.0,3.0,1.500,2.0,4.000,2.5,2.5,mas_with_evaluator


## Stage 5b - Evaluation Contour (Reproducible Metrics Registry)

Primary evaluation contour: metric groups, per-topic registration table (PDF format), MLflow summary.


In [10]:
from IPython.display import display, Markdown

METRIC_GROUPS = pd.DataFrame([
    {"group": "Качество результата", "metrics": "eval_correctness, eval_groundedness, eval_completeness, schema_coverage, eval_field_coverage"},
    {"group": "Качество процесса", "metrics": "steps, unnecessary_actions"},
    {"group": "Эксплуатационные", "metrics": "latency_sec, tool_errors, fallback_activations, timeouts"},
    {"group": "Сводная", "metrics": "evaluator_overall"},
])
display(METRIC_GROUPS)

main_exp = results_df[results_df["stage"] == "exp_main"].copy()

def config_label(row: pd.Series) -> str:
    if row["variant"] == "baseline":
        return "baseline"
    if row["variant"] == "mas_with_evaluator":
        return "agent + evaluator"
    return "agent"

results_registry_df = main_exp.assign(Configuration=main_exp.apply(config_label, axis=1))[
    ["topic", "Configuration", "eval_correctness", "eval_groundedness", "eval_completeness",
     "steps", "latency_sec", "tool_errors", "evaluator_overall"]
].rename(columns={
    "topic": "Topic",
    "eval_correctness": "Correctness",
    "eval_groundedness": "Groundedness",
    "eval_completeness": "Completeness",
    "steps": "Steps",
    "latency_sec": "Latency, s",
    "tool_errors": "Tool errors",
    "evaluator_overall": "Rubric",
})
print(f"Registration table rows: {len(results_registry_df)} (expected 24)")
display(results_registry_df)

summary_metrics = {}
for variant in ["baseline", "mas", "mas_with_evaluator"]:
    sub = main_exp[main_exp["variant"] == variant]
    if sub.empty:
        continue
    for col in ["eval_correctness", "eval_groundedness", "eval_completeness", "evaluator_overall",
                "steps", "latency_sec", "tool_errors", "unnecessary_actions", "fallback_activations"]:
        if col in sub.columns:
            summary_metrics[f"{variant}_{col}_mean"] = float(sub[col].mean())
with mlflow.start_run(run_name="evaluation_summary"):
    mlflow.set_tag("stage", "evaluation_contour")
    mlflow.log_params({"ollama_model": OLLAMA_MODEL, "replay_only": int(REPLAY_ONLY)})
    for k, v in summary_metrics.items():
        mlflow.log_metric(k, v)
    reg_path = RESULTS_CACHE_DIR / "results_registry.csv"
    results_registry_df.to_csv(reg_path, index=False)
    mlflow.log_artifact(str(reg_path))
print("Logged evaluation summary to MLflow")


,group,metrics
0,Качество результата,"eval_correctness, eval_groundedness, eval_comp..."
1,Качество процесса,"steps, unnecessary_actions"
2,Эксплуатационные,"latency_sec, tool_errors, fallback_activations..."
3,Сводная,evaluator_overall


Registration table rows: 24 (expected 24)


,Topic,Configuration,Correctness,Groundedness,Completeness,Steps,"Latency, s",Tool errors,Rubric
0,Constitutional AI,baseline,4.0,4.0,3.667,3,15.841711,0,3.7
1,Constitutional AI,agent,4.5,4.0,4.333,10,15.636610,0,4.1
2,Constitutional AI,agent + evaluator,2.0,3.0,1.500,10,14.697098,0,2.5
9,causal inference,baseline,5.0,4.0,5.000,3,20.710432,0,4.5
10,causal inference,agent,5.0,4.0,5.000,10,25.456862,0,4.5
11,causal inference,agent + evaluator,3.8,4.2,3.500,10,24.636992,0,4.0
18,meta-learning,baseline,3.5,4.0,3.000,3,13.445667,0,3.3
19,meta-learning,agent,3.5,4.0,3.000,10,14.399045,0,3.3
20,meta-learning,agent + evaluator,2.0,3.0,1.000,10,13.024553,0,2.8
27,neuro-symbolic artificial intelligence,baseline,5.0,4.0,5.000,3,19.577883,0,4.5


Logged evaluation summary to MLflow


## Stage 6 - Analytics, Ablations, and Failure Cases

**Control question:** Do plots cover all three main configurations?


In [11]:
main_df = results_df[results_df["stage"] == "exp_main"].copy()
quality_cols = ["schema_coverage", "evaluator_overall", "eval_correctness", "eval_groundedness", "eval_completeness"]
process_cols = ["steps", "latency_sec", "unnecessary_actions", "source_count"]
ops_cols = ["tool_calls", "tool_errors", "fallback_activations", "timeouts"]

quality_summary = main_df.groupby("variant", as_index=False)[quality_cols].mean(numeric_only=True)
process_summary = main_df.groupby("variant", as_index=False)[process_cols].mean(numeric_only=True)
ops_summary = main_df.groupby("variant", as_index=False)[ops_cols].mean(numeric_only=True)

fig_q = px.bar(quality_summary.melt(id_vars=["variant"], value_vars=quality_cols), x="variant", y="value", color="variable", barmode="group", title="Quality Metrics by Variant", template="plotly_white")
fig_p = px.bar(process_summary.melt(id_vars=["variant"], value_vars=process_cols), x="variant", y="value", color="variable", barmode="group", title="Process Metrics by Variant", template="plotly_white")
fig_o = px.bar(ops_summary.melt(id_vars=["variant"], value_vars=ops_cols), x="variant", y="value", color="variable", barmode="group", title="Operational Reliability Metrics", template="plotly_white")
fig_q.show(); fig_p.show(); fig_o.show()

report_table_main = main_df.groupby("variant", as_index=False)[
    ["eval_correctness", "eval_groundedness", "eval_completeness", "schema_coverage", "steps", "latency_sec", "tool_errors", "evaluator_overall"]
].mean(numeric_only=True)
display(report_table_main)

per_topic_table = main_df.pivot_table(index="topic", columns="variant", values="evaluator_overall", aggfunc="mean")
display(per_topic_table)

src_ab = results_df[results_df["stage"] == "ablation_sources"].groupby("source_limit", as_index=False)["evaluator_overall"].mean()
step_ab = results_df[results_df["stage"] == "ablation_steps"].groupby("max_steps", as_index=False)["evaluator_overall"].mean()
px.line(src_ab, x="source_limit", y="evaluator_overall", markers=True, title="Ablation: source_limit vs rubric", template="plotly_white").show()
px.line(step_ab, x="max_steps", y="evaluator_overall", markers=True, title="Ablation: max_steps vs rubric", template="plotly_white").show()


,variant,eval_correctness,eval_groundedness,eval_completeness,schema_coverage,steps,latency_sec,tool_errors,evaluator_overall
0,baseline,4.4375,4.000,4.250000,0.812500,3.0,18.246853,0.0,4.0375
1,mas,4.5625,4.000,4.416625,0.854167,10.0,20.333711,0.0,4.1250
2,mas_with_evaluator,3.1750,3.575,2.812500,0.854167,10.0,19.052309,0.0,3.4000


variant,baseline,mas,mas_with_evaluator
topic,,,
Constitutional AI,3.7,4.1,2.5
causal inference,4.5,4.5,4.0
federated learning,4.3,4.3,4.1
knowledge graph embedding,3.7,3.7,2.5
meta-learning,3.3,3.3,2.8
mixture of experts,4.2,4.1,4.1
neuro-symbolic artificial intelligence,4.5,4.5,4.4
reinforcement learning from human feedback,4.1,4.5,2.8


In [12]:
agent_vs_eval = report_table_main.set_index("variant").loc[["mas", "mas_with_evaluator"]]
agent_delta_table = (agent_vs_eval.loc["mas_with_evaluator"] - agent_vs_eval.loc["mas"]).to_frame("delta_evaluator_minus_mas")
display(agent_delta_table)

def failure_type_from_row(row: pd.Series) -> str:
    if row.get("tool_errors", 0) > 0:
        return "weak_search"
    if row.get("schema_coverage", 0) < 0.95:
        return "loss_of_completeness"
    if row.get("unnecessary_actions", 0) > 2:
        return "unnecessary_steps"
    return "weak_source_to_conclusion_linkage"

def load_trace_for_row(row: pd.Series) -> List[Dict[str, Any]]:
    cfg = ExperimentConfig(
        stage=row["stage"], topic=row["topic"], mode=row["mode"],
        source_limit=int(row["source_limit"]), max_steps=int(row["max_steps"]),
        use_evaluator=bool(row["use_evaluator"]), llm=BASE_LLM_CFG,
    )
    data = load_result(cfg)
    return data.get("trace", []) if data else []

weak_rows = main_df.sort_values(by=["evaluator_overall", "schema_coverage"], ascending=True).head(3)
manual_cases = []
for _, row in weak_rows.iterrows():
    trace = load_trace_for_row(row)
    critical = trace[-1] if trace else {}
    manual_cases.append({
        "topic": row["topic"],
        "variant": row["variant"],
        "failure_step": critical.get("action", "unknown"),
        "failure_type": failure_type_from_row(row),
        "root_cause": "Weak retrieval or incomplete synthesis before supervisor stop.",
        "mitigation": "Increase source_limit, tighten filtering, enable evaluator loop.",
        "trace_excerpt": json.dumps(critical, ensure_ascii=False)[:300],
    })
manual_failure_cases_df = pd.DataFrame(manual_cases)
display(manual_failure_cases_df)


,delta_evaluator_minus_mas
eval_correctness,-1.387500
eval_groundedness,-0.425000
eval_completeness,-1.604125
schema_coverage,0.000000
steps,0.000000
latency_sec,-1.281402
tool_errors,0.000000
evaluator_overall,-0.725000


,topic,variant,failure_step,failure_type,root_cause,mitigation,trace_excerpt
0,knowledge graph embedding,mas_with_evaluator,check_stop,loss_of_completeness,Weak retrieval or incomplete synthesis before ...,"Increase source_limit, tighten filtering, enab...","{""step_id"": 10, ""actor"": ""supervisor"", ""action..."
1,Constitutional AI,mas_with_evaluator,check_stop,loss_of_completeness,Weak retrieval or incomplete synthesis before ...,"Increase source_limit, tighten filtering, enab...","{""step_id"": 10, ""actor"": ""supervisor"", ""action..."
2,meta-learning,mas_with_evaluator,check_stop,loss_of_completeness,Weak retrieval or incomplete synthesis before ...,"Increase source_limit, tighten filtering, enab...","{""step_id"": 10, ""actor"": ""supervisor"", ""action..."


## Stage 7 - Architecture Comparison: Baseline vs Custom MAS

**Control question:** What does MAS buy beyond baseline in controllability and trace observability?


In [13]:
arch_compare = pd.DataFrame([
    {"dimension": "steps (mean)", "baseline": report_table_main.loc[report_table_main.variant == "baseline", "steps"].values[0] if "baseline" in report_table_main.variant.values else 0,
     "mas": report_table_main.loc[report_table_main.variant == "mas", "steps"].values[0] if "mas" in report_table_main.variant.values else 0,
     "mas_with_evaluator": report_table_main.loc[report_table_main.variant == "mas_with_evaluator", "steps"].values[0] if "mas_with_evaluator" in report_table_main.variant.values else 0},
    {"dimension": "latency_sec (mean)", "baseline": report_table_main.loc[report_table_main.variant == "baseline", "latency_sec"].values[0] if "baseline" in report_table_main.variant.values else 0,
     "mas": report_table_main.loc[report_table_main.variant == "mas", "latency_sec"].values[0] if "mas" in report_table_main.variant.values else 0,
     "mas_with_evaluator": report_table_main.loc[report_table_main.variant == "mas_with_evaluator", "latency_sec"].values[0] if "mas_with_evaluator" in report_table_main.variant.values else 0},
    {"dimension": "evaluator_overall (mean)", "baseline": report_table_main.loc[report_table_main.variant == "baseline", "evaluator_overall"].values[0] if "baseline" in report_table_main.variant.values else 0,
     "mas": report_table_main.loc[report_table_main.variant == "mas", "evaluator_overall"].values[0] if "mas" in report_table_main.variant.values else 0,
     "mas_with_evaluator": report_table_main.loc[report_table_main.variant == "mas_with_evaluator", "evaluator_overall"].values[0] if "mas_with_evaluator" in report_table_main.variant.values else 0},
    {"dimension": "tool_errors (mean)", "baseline": report_table_main.loc[report_table_main.variant == "baseline", "tool_errors"].values[0] if "baseline" in report_table_main.variant.values else 0,
     "mas": report_table_main.loc[report_table_main.variant == "mas", "tool_errors"].values[0] if "mas" in report_table_main.variant.values else 0,
     "mas_with_evaluator": report_table_main.loc[report_table_main.variant == "mas_with_evaluator", "tool_errors"].values[0] if "mas_with_evaluator" in report_table_main.variant.values else 0},
])
display(arch_compare)
px.bar(arch_compare.melt(id_vars=["dimension"], value_vars=["baseline", "mas", "mas_with_evaluator"]),
       x="dimension", y="value", color="variable", barmode="group",
       title="Baseline vs MAS vs MAS+Evaluator", template="plotly_white").show()


,dimension,baseline,mas,mas_with_evaluator
0,steps (mean),3.000000,10.000000,10.000000
1,latency_sec (mean),18.246853,20.333711,19.052309
2,evaluator_overall (mean),4.037500,4.125000,3.400000
3,tool_errors (mean),0.000000,0.000000,0.000000


## Stage 8 - Final Research Report (In-Notebook)

Rich display: narrative markdown + tables + reference to Stage 6 figures.


In [15]:
def _pct_delta(new: float, old: float) -> str:
    if old == 0:
        return "n/a"
    return f"{100 * (new - old) / old:+.1f}%"

def build_report_sections() -> List[Tuple[str, Optional[pd.DataFrame]]]:
    rt = report_table_main.set_index("variant")
    b, m, me = "baseline", "mas", "mas_with_evaluator"
    sections = []
    sections.append(('# Final Research Report\n\n## 1. Introduction and task\n'
        'Compare baseline one-shot retrieval+synthesis vs custom MAS with optional LLM evaluator on 8 topics.\n\n'
        '## 2. Architecture\n'
        '- Baseline: Wikipedia + OpenAlex -> single Ollama synthesis (deterministic fallback).\n'
        '- MAS: staged tools, filtering, supervisor stops, Ollama synthesis.\n'
        '- Evaluator: Ollama JSON rubric when use_evaluator=True; deterministic otherwise.\n'
        '- State: AgentState trace with payload and source/note deltas.', None))
    sections.append(('## 3. Evaluation methodology\n'
        'Metric groups: result quality, process quality, operational, summary rubric.', METRIC_GROUPS))
    sections.append(('## 4. Experiment design\n'
        '8 topics x {baseline, agent, agent+evaluator}; ablations source_limit and max_steps.', None))
    sections.append(('## 5. Results\n\n### 5.1 Per-topic registration table', results_registry_df))
    sections.append(('### 5.2 Aggregated comparison (three configurations)', report_table_main))
    if b in rt.index and m in rt.index:
        rub_d = float(rt.loc[m, "evaluator_overall"] - rt.loc[b, "evaluator_overall"])
        lat_d = _pct_delta(float(rt.loc[m, "latency_sec"]), float(rt.loc[b, "latency_sec"]))
        st_d = _pct_delta(float(rt.loc[m, "steps"]), float(rt.loc[b, "steps"]))
        sections.append((f'### 5.3 Quality vs process tradeoff\n'
            f'Baseline vs agent: rubric delta={rub_d:+.2f}; latency {lat_d}; steps {st_d}. '
            'Weigh rubric gains against latency/step cost.', agent_delta_table))
    if m in rt.index and me in rt.index:
        rub2 = float(rt.loc[me, "evaluator_overall"] - rt.loc[m, "evaluator_overall"])
        lat2 = _pct_delta(float(rt.loc[me, "latency_sec"]), float(rt.loc[m, "latency_sec"]))
        sections.append((f'Agent vs agent+evaluator: rubric delta={rub2:+.2f}; latency {lat2}.', None))
    sections.append(('### 5.4 Ablation\nSee Stage 6 line plots for source_limit and max_steps.', None))
    sections.append(('## 6. Failure case studies (>=3)', manual_failure_cases_df))
    best = rt["evaluator_overall"].idxmax() if "evaluator_overall" in rt.columns else "mas"
    why = []
    if best != b and float(rt.loc[best, "eval_completeness"]) > float(rt.loc[b, "eval_completeness"]):
        why.append("higher completeness")
    if best != b and float(rt.loc[best, "eval_groundedness"]) > float(rt.loc[b, "eval_groundedness"]):
        why.append("better grounding")
    if best != b and float(rt.loc[best, "tool_errors"]) <= float(rt.loc[b, "tool_errors"]):
        why.append("similar or fewer tool errors")
    if not why:
        why.append("best mean rubric in this run")
    sections.append((f'## 7. Final conclusion\nRecommended variant: {best} because {", ".join(why)}.\n\n'
        f'## 8. Reproducibility\nModel {OLLAMA_MODEL} at {OLLAMA_BASE_URL}; REPLAY_ONLY={REPLAY_ONLY}.', None))
    return sections

def render_notebook_report(sections):
    for md_text, df in sections:
        display(Markdown(md_text))
        if df is not None:
            display(df)

report_sections = build_report_sections()
render_notebook_report(report_sections)

report_path = RESULTS_CACHE_DIR / "lab3_report.md"
report_path.write_text("\n\n".join(md for md, _ in report_sections), encoding="utf-8")
with mlflow.start_run(run_name="final_report"):
    mlflow.log_artifact(str(report_path))
print(f"Optional export: {report_path}")


# Final Research Report

## 1. Introduction and task
Compare baseline one-shot retrieval+synthesis vs custom MAS with optional LLM evaluator on 8 topics.

## 2. Architecture
- Baseline: Wikipedia + OpenAlex -> single Ollama synthesis (deterministic fallback).
- MAS: staged tools, filtering, supervisor stops, Ollama synthesis.
- Evaluator: Ollama JSON rubric when use_evaluator=True; deterministic otherwise.
- State: AgentState trace with payload and source/note deltas.

## 3. Evaluation methodology
Metric groups: result quality, process quality, operational, summary rubric.

,group,metrics
0,Качество результата,"eval_correctness, eval_groundedness, eval_comp..."
1,Качество процесса,"steps, unnecessary_actions"
2,Эксплуатационные,"latency_sec, tool_errors, fallback_activations..."
3,Сводная,evaluator_overall


## 4. Experiment design
8 topics x {baseline, agent, agent+evaluator}; ablations source_limit and max_steps.

## 5. Results

### 5.1 Per-topic registration table

...

### 5.4 Ablation
See Stage 6 line plots for source_limit and max_steps.

## 6. Failure case studies (>=3)

,topic,variant,failure_step,failure_type,root_cause,mitigation,trace_excerpt
0,knowledge graph embedding,mas_with_evaluator,check_stop,loss_of_completeness,Weak retrieval or incomplete synthesis before ...,"Increase source_limit, tighten filtering, enab...","{""step_id"": 10, ""actor"": ""supervisor"", ""action..."
1,Constitutional AI,mas_with_evaluator,check_stop,loss_of_completeness,Weak retrieval or incomplete synthesis before ...,"Increase source_limit, tighten filtering, enab...","{""step_id"": 10, ""actor"": ""supervisor"", ""action..."
2,meta-learning,mas_with_evaluator,check_stop,loss_of_completeness,Weak retrieval or incomplete synthesis before ...,"Increase source_limit, tighten filtering, enab...","{""step_id"": 10, ""actor"": ""supervisor"", ""action..."


## 7. Final conclusion
Recommended variant: mas because higher completeness, similar or fewer tool errors.

## 8. Reproducibility
Model llama3.1 at http://127.0.0.1:11434; REPLAY_ONLY=False.

Optional export: /Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/lab-3/results_cache/lab3_report.md
